# 02b — Encounter features

Compute previous encounters and time since the last admission. Duplicate spells are reduced before feature generation.

In [ ]:
%load_ext autoreload
%autoreload 2

In [6]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define data path
DATA_DIR = get_path('data/synthetic/10000p') # Manually defined path
DATA_DIR = get_data_root_from_schema()       # Paramter schema_path= / default_dir=      
DATA_PROC_DIR = DATA_DIR / 'processed' 
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [ ]:
# --- OPTION A: Load using pandas ---
def load_from_pandas(DATA_DIR):
    return pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")

df_episodes = load_from_pandas(DATA_DIR)
df_episodes = keep_first_spell(df_episodes)
df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])

In [ ]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
schema_cfg = 'configs/schema.yaml'
phenotype_cfg = 'configs/phenotypes.yaml'
domains = ['episodes']

# Load dataframes
dfs = load_from_duckdb(
    schema_path=schema_cfg,
    phenotype_path=phenotype_cfg,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])

In [ ]:
df_episodes = keep_first_spell(df_episodes)
df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])

In [7]:
from icare_risk.clinphen.utils import (
    compute_previous_encounters,
    compute_time_since_last_admission
)

df_encounters = compute_previous_encounters(df_episodes, days=365)
df_encounters = compute_time_since_last_admission(
    df_episodes=df_encounters,
    subject_col="SUBJECT",
    spell_col="SPELL_IDENTIFIER",
    admission_col="ADMISSION_DATE",
    discharge_col="DISCHARGE_DATE",
    use_discharge_time=True,
)
df_encounters = keep_first_spell(df_encounters, "encounter features")
display(df_encounters.head())


episodes: 10,000 rows -> 10,000 unique spells
encounter features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,index_admission,num_encounters_last_365d,days_since_last_discharge
0,10001,7867287,2998828,3749211,2023-02-12,00:52:00,64,2023-02-26,140,oral surgery,1,2023-02-12 00:52:00,0,NaN
1,10002,6551735,8909338,9353593,2022-11-16,23:43:00,72,2022-12-01,320,cardiology,1,2022-11-16 23:43:00,0,NaN
2,10003,4441228,1307783,3518105,2022-05-12,20:10:00,29,2022-05-27,320,cardiology,unknown,2022-05-12 20:10:00,0,NaN
3,10004,5593202,5695692,6593442,2023-10-11,00:52:00,19,2023-10-30,300,general medicine,4,2023-10-11 00:52:00,0,NaN
4,10005,6532182,7455884,6203572,2023-08-14,05:42:00,50,2023-08-18,300,general medicine,7,2023-08-14 05:42:00,0,NaN


In [8]:
filename = "02c_derived_features_encounters.parquet"
output_path = DATA_PROC_DIR / filename
df_encounters.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/data/synthetic/10000p/processed/02c_derived_features_encounters.parquet
